# 可信井高频残差分解：第一轮高斯尺度空间

本 notebook 实现 `WELL_LOG_HIGH_FREQUENCY_RESIDUAL_EXPERIMENT.md` 的第一轮。

目标是直接观察：

```text
full well log-AI
= Gaussian recoverable log-AI
+ high-frequency residual
```

本轮比较 `0.75T / 1.00T / 1.50T` 三个平滑尺度。它们只是形态基线，不代表已经证明了地震可恢复频带。主要结果是逐井曲线、局部放大图、周期性诊断和人工评价模板。

输入采用第五步发布的深度平移全频 LAS；井震标定滤波曲线不参与残差生成。


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from scipy import signal
from scipy.ndimage import gaussian_filter1d

repo_root = Path.cwd().resolve()
if not (repo_root / "src").is_dir():
    repo_root = repo_root.parent
if not (repo_root / "src").is_dir():
    raise RuntimeError("Could not locate repository root containing src/.")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from cup.petrel.load import import_well_tops_petrel
from cup.physics.numpy_backend import forward_depth, velocity_from_ai
from cup.synthetic.core.signal import finite_support_fir, valid_filter_decimate
from cup.well.assets import normalize_well_name
from cup.well.las import read_las_curve

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 180,
        "axes.grid": True,
        "grid.alpha": 0.22,
        "font.size": 9,
    }
)
print(f"Repository: {repo_root}")

In [ ]:
# 所有可修改参数集中在本 cell。
RUN_ID = "20260810_gaussian_scale_space"
OUTPUT_DIR = repo_root / "experiments" / "well_residual_decomposition" / "results" / RUN_ID

BATCH_DIR = repo_root / "scripts" / "output" / "wavelet_batch_synthetic_depth_20260719_172510"
METRICS_FILE = BATCH_DIR / "wavelet_batch_metrics.csv"
WELL_TOPS_FILE = repo_root / "data" / "raw" / "well_tops"
FORWARD_INPUTS_FILE = (
    repo_root / "scripts" / "output" / "depth_forward_model_inputs_20260719_172553" / "forward_model_inputs.json"
)

TRUSTED_WELLS = (
    "2-ANP-2A-RJS",
    "L1-NW1",
    "L5-NW5",
    "L9-NW4A",
    "NW11",
    "NW8",
)
DIAGNOSTIC_WELLS = ("L3-NW2A", "L6-NW3A", "NW7")
INCLUDE_DIAGNOSTIC_WELLS = False

HORIZON_SURFACES = {
    "base_of_salt": "BVE100_TOP",
    "base_of_bve": "ITP_TOP",
    "base_of_itp": "ITP_BOT",
}
OPTIONAL_HORIZONS_BY_WELL = {
    "NW8": {"base_of_itp"},
}

# 当前子波频谱较宽，单个最大谱峰对 FFT 频率格点较敏感。
# 默认使用能量中位频率，并同时发布 peak/centroid 供审计。
REFERENCE_FREQUENCY_STATISTIC = "energy_median"
FWHM_MULTIPLIERS = {
    "G075": 0.75,
    "G100": 1.00,
    "G150": 1.50,
}
MIN_RUN_FWHM_MULTIPLE = 1.0
EDGE_DOMINATED_BELOW_FWHM_MULTIPLE = 4.0
PLOT_CONTEXT_M = 40.0
ZOOM_WINDOW_M = 120.0
MODEL_GRID_INTERVAL_M = 5.0
FORWARD_OUTPUT_CHUNK_SIZE = 32

for required in (METRICS_FILE, WELL_TOPS_FILE, FORWARD_INPUTS_FILE):
    if not required.exists():
        raise FileNotFoundError(required)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "figures").mkdir(exist_ok=True)
(OUTPUT_DIR / "wells").mkdir(exist_ok=True)

print(f"Output: {OUTPUT_DIR}")

## 1. 加载井、层位、子波和正演合同

第五步的 shifted LAS 仍以 shifted MD 作为 LAS index。当前井均为直井，因此 notebook 显式执行：

```text
TVDSS = shifted_MD - KB
```

层位来自 Petrel well tops。层位原始 TVDSS 使用同一口井的深度平移曲线转换到 shifted TVDSS。


In [ ]:
batch_metrics = pd.read_csv(METRICS_FILE)
if batch_metrics["well_name"].duplicated().any():
    raise ValueError("wavelet_batch_metrics.csv contains duplicate well_name rows.")
batch_metrics = batch_metrics.set_index("well_name", drop=False)

well_tops = import_well_tops_petrel(WELL_TOPS_FILE)
forward_inputs = json.loads(FORWARD_INPUTS_FILE.read_text(encoding="utf-8"))


def resolve_repo_path(value):
    path = Path(str(value))
    return path if path.is_absolute() else repo_root / path


wavelet_path = resolve_repo_path(forward_inputs["wavelet"]["path"])
wavelet_frame = pd.read_csv(wavelet_path)
wavelet_time_s = wavelet_frame["time_s"].to_numpy(dtype=np.float64)
wavelet_amplitude = wavelet_frame["amplitude"].to_numpy(dtype=np.float64)

relation = forward_inputs["ai_velocity_relation"]
AI_VP_A = float(relation["a"])
AI_VP_B = float(relation["b"])


def wavelet_frequency_summary(time_s, amplitude):
    time_s = np.asarray(time_s, dtype=np.float64)
    amplitude = np.asarray(amplitude, dtype=np.float64)
    dt_s = float(np.median(np.diff(time_s)))
    centered = amplitude - np.mean(amplitude)
    frequency = np.fft.rfftfreq(centered.size, d=dt_s)
    power = np.abs(np.fft.rfft(centered)) ** 2
    power[0] = 0.0
    if not np.any(power > 0.0):
        raise ValueError("Wavelet has no non-zero spectral energy.")
    probability = power / np.sum(power)
    cumulative = np.cumsum(probability)
    quantile = lambda q: float(frequency[np.searchsorted(cumulative, q)])
    return {
        "dt_s": dt_s,
        "peak_hz": float(frequency[int(np.argmax(power))]),
        "energy_centroid_hz": float(np.sum(frequency * probability)),
        "energy_p10_hz": quantile(0.10),
        "energy_median_hz": quantile(0.50),
        "energy_p90_hz": quantile(0.90),
    }


wavelet_summary = wavelet_frequency_summary(wavelet_time_s, wavelet_amplitude)
frequency_key = {
    "peak": "peak_hz",
    "energy_centroid": "energy_centroid_hz",
    "energy_median": "energy_median_hz",
}[REFERENCE_FREQUENCY_STATISTIC]
reference_frequency_hz = float(wavelet_summary[frequency_key])

display(pd.DataFrame([wavelet_summary]))
print(f"Reference frequency: {reference_frequency_hz:.3f} Hz ({REFERENCE_FREQUENCY_STATISTIC})")

In [ ]:
def finite_runs(mask):
    mask = np.asarray(mask, dtype=bool)
    padded = np.concatenate(([False], mask, [False]))
    changes = np.flatnonzero(padded[1:] != padded[:-1])
    return tuple(slice(int(start), int(stop)) for start, stop in changes.reshape(-1, 2))


def shifted_horizon_tvdss(well_name, kb_m, shift_curve_path):
    curve = pd.read_csv(shift_curve_path)
    source_depth = curve["tvdss_m"].to_numpy(dtype=np.float64)
    shift_m = curve["depth_shift_m"].to_numpy(dtype=np.float64)
    if np.any(~np.isfinite(source_depth)) or np.any(np.diff(source_depth) <= 0.0):
        raise ValueError(f"Invalid depth shift curve for {well_name}.")

    well_key = normalize_well_name(well_name)
    horizons = {}
    for horizon_name, surface_name in HORIZON_SURFACES.items():
        selected = well_tops[
            well_tops["Well"].map(normalize_well_name).eq(well_key)
            & well_tops["Surface"].astype(str).str.strip().str.casefold().eq(surface_name.casefold())
        ]
        if len(selected) == 0 and horizon_name in OPTIONAL_HORIZONS_BY_WELL.get(well_name, set()):
            continue
        if len(selected) != 1:
            raise ValueError(f"{well_name}: expected exactly one well top {surface_name}, found {len(selected)}.")
        md_m = float(pd.to_numeric(selected.iloc[0]["MD"]))
        original_tvdss_m = md_m - float(kb_m)
        local_shift_m = float(np.interp(original_tvdss_m, source_depth, shift_m))
        horizons[horizon_name] = original_tvdss_m + local_shift_m
    values = np.asarray(list(horizons.values()), dtype=np.float64)
    if np.any(np.diff(values) <= 0.0):
        raise ValueError(f"{well_name}: shifted horizons are not strictly ordered.")
    return horizons


def load_well(well_name):
    if well_name not in batch_metrics.index:
        raise KeyError(f"Well {well_name!r} is absent from wavelet_batch_metrics.csv.")
    row = batch_metrics.loc[well_name]
    if str(row["status"]) != "ok":
        raise ValueError(f"Well {well_name!r} has non-ok batch status.")

    kb_m = float(row["kb_m"])
    las_path = resolve_repo_path(row["shifted_preprocessed_las_path"])
    shift_curve_path = resolve_repo_path(row["depth_shift_curve_path"])
    if not las_path.exists() or not shift_curve_path.exists():
        raise FileNotFoundError(f"Missing LAS or shift curve for {well_name}.")

    ai_log = read_las_curve(las_path, "AI", match_policy="exact")
    shifted_md_m = np.asarray(ai_log.basis, dtype=np.float64)
    ai = np.asarray(ai_log.values, dtype=np.float64)
    tvdss_m = shifted_md_m - kb_m
    valid = np.isfinite(ai) & (ai > 0.0)
    log_ai = np.full(ai.shape, np.nan, dtype=np.float64)
    log_ai[valid] = np.log(ai[valid])

    intervals = np.diff(tvdss_m)
    dz_m = float(np.median(intervals))
    if np.any(intervals <= 0.0) or not np.allclose(intervals, dz_m, rtol=1e-5, atol=1e-6):
        raise ValueError(f"{well_name}: shifted LAS axis is not regularly increasing.")

    median_vp_mps = float(row["median_vp_mps"])
    tuning_scale_m = median_vp_mps / (4.0 * reference_frequency_hz)
    horizons = shifted_horizon_tvdss(well_name, kb_m, shift_curve_path)
    return {
        "well_name": well_name,
        "tvdss_m": tvdss_m,
        "log_ai": log_ai,
        "valid": valid,
        "dz_m": dz_m,
        "median_vp_mps": median_vp_mps,
        "tuning_scale_m": tuning_scale_m,
        "horizons": horizons,
        "tie_corr": float(row["corr"]),
        "las_path": str(las_path),
        "shift_curve_path": str(shift_curve_path),
    }


selected_wells = TRUSTED_WELLS + (DIAGNOSTIC_WELLS if INCLUDE_DIAGNOSTIC_WELLS else ())
wells = {name: load_well(name) for name in selected_wells}

well_inventory = pd.DataFrame(
    [
        {
            "well_name": item["well_name"],
            "tie_corr": item["tie_corr"],
            "dz_m": item["dz_m"],
            "median_vp_mps": item["median_vp_mps"],
            "tuning_scale_m": item["tuning_scale_m"],
            "valid_samples": int(np.count_nonzero(item["valid"])),
            **{f"{name}_tvdss_m": value for name, value in item["horizons"].items()},
        }
        for item in wells.values()
    ]
)
display(well_inventory)

## 2. 高斯尺度空间分解

每个连续有效井段独立做零相位高斯平滑，边缘使用 reflect extension。长度不足 `1 × FWHM` 的连续段保持 unsupported；介于 `1–4 × FWHM` 的结果明确标记为 edge-dominated。三档结果严格使用同一输入曲线。

残差曲线始终保留在 0.1 m 井轴；正演诊断先使用 Synthoseis-lite 的有限支撑算子投影到 5 m 模型网格，再调用 depth forward。


In [ ]:
def gaussian_decomposition(well, fwhm_m):
    values = well["log_ai"]
    dz_m = float(well["dz_m"])
    sigma_samples = (float(fwhm_m) / 2.354820045) / dz_m
    minimum_samples = int(np.ceil(MIN_RUN_FWHM_MULTIPLE * float(fwhm_m) / dz_m))

    recoverable = np.full(values.shape, np.nan, dtype=np.float64)
    support = np.zeros(values.shape, dtype=bool)
    for run in finite_runs(well["valid"]):
        if run.stop - run.start < minimum_samples:
            continue
        recoverable[run] = gaussian_filter1d(
            values[run],
            sigma=sigma_samples,
            mode="reflect",
            truncate=4.0,
        )
        support[run] = True

    residual = np.where(support, values - recoverable, np.nan)
    reconstructed = np.where(support, recoverable + residual, np.nan)
    valid_run_lengths_m = np.asarray(
        [(run.stop - run.start) * dz_m for run in finite_runs(well["valid"])],
        dtype=np.float64,
    )
    longest_run_ratio = float(np.max(valid_run_lengths_m) / float(fwhm_m)) if valid_run_lengths_m.size else 0.0
    return {
        "fwhm_m": float(fwhm_m),
        "sigma_samples": float(sigma_samples),
        "support": support,
        "recoverable_log_ai": recoverable,
        "high_frequency_residual": residual,
        "reconstructed_log_ai": reconstructed,
        "longest_valid_run_fwhm_ratio": longest_run_ratio,
        "edge_dominated": bool(longest_run_ratio < EDGE_DOMINATED_BELOW_FWHM_MULTIPLE),
    }


def projected_forward_log_ai(depth_m, log_ai, support, highres_interval_m):
    depth_m = np.asarray(depth_m, dtype=np.float64)
    log_ai = np.asarray(log_ai, dtype=np.float64)
    support = np.asarray(support, dtype=bool)
    output = np.full(log_ai.shape, np.nan, dtype=np.float64)
    factor_float = MODEL_GRID_INTERVAL_M / float(highres_interval_m)
    factor = int(round(factor_float))
    if factor < 1 or not np.isclose(factor_float, factor, rtol=0.0, atol=1e-6):
        raise ValueError("High-resolution LAS axis is not nested with the 5 m model interval.")
    taps = finite_support_fir(factor)
    for run in finite_runs(support & np.isfinite(log_ai)):
        if run.stop - run.start < 3:
            continue
        local_log_ai = log_ai[run]
        model_log_ai, model_support = valid_filter_decimate(
            local_log_ai,
            factor=factor,
            taps=taps,
        )
        model_depth = depth_m[run][::factor]
        model_indices = np.arange(run.start, run.stop, factor, dtype=np.int64)
        if model_log_ai.shape != model_depth.shape or model_depth.shape != model_indices.shape:
            raise ValueError("Projected model arrays have inconsistent shapes.")
        if model_log_ai.size < 2 or not np.any(model_support):
            continue
        local_ai = np.exp(model_log_ai)
        local_vp = velocity_from_ai(local_ai, a=AI_VP_A, b=AI_VP_B)
        if np.any(~np.isfinite(local_vp)) or np.any(local_vp <= 0.0):
            raise ValueError("AI-Vp relation produced invalid velocity.")
        synthetic = forward_depth(
            model_log_ai,
            local_vp,
            model_depth,
            wavelet_time_s,
            wavelet_amplitude,
            output_chunk_size=FORWARD_OUTPUT_CHUNK_SIZE,
        )
        output[model_indices[model_support]] = synthetic[model_support]
    return output


def longest_supported_run(support):
    runs = finite_runs(support)
    return max(runs, key=lambda item: item.stop - item.start) if runs else None


def periodicity_profile(residual, support, dz_m, max_lag_m):
    run = longest_supported_run(support & np.isfinite(residual))
    if run is None or run.stop - run.start < 32:
        return None
    values = np.asarray(residual[run], dtype=np.float64)
    values = signal.detrend(values, type="linear")
    if np.std(values) <= 0.0:
        return None

    nperseg = min(2048, values.size)
    frequency, power = signal.welch(
        values,
        fs=1.0 / float(dz_m),
        nperseg=nperseg,
        detrend="linear",
        scaling="density",
    )
    positive = frequency > 0.0
    frequency = frequency[positive]
    power = power[positive]
    power_probability = power / np.sum(power)

    centered = values - np.mean(values)
    autocorrelation = signal.correlate(centered, centered, mode="full", method="fft")
    autocorrelation = autocorrelation[centered.size - 1 :]
    overlap = np.arange(centered.size, 0, -1, dtype=np.float64)
    autocorrelation = autocorrelation / overlap
    autocorrelation = autocorrelation / autocorrelation[0]
    lag_m = np.arange(autocorrelation.size, dtype=np.float64) * float(dz_m)
    keep = lag_m <= float(max_lag_m)

    signs = np.signbit(centered)
    crossings = np.flatnonzero(signs[1:] != signs[:-1])
    crossing_interval_m = np.diff(crossings).astype(np.float64) * float(dz_m)

    spectral_entropy = -float(np.sum(power_probability * np.log(np.clip(power_probability, 1e-15, 1.0)))) / np.log(
        max(power_probability.size, 2)
    )
    window = min(5, power_probability.size)
    peak_fraction = float(np.max(np.convolve(power_probability, np.ones(window), mode="same")))

    peak_start = max(1, int(np.ceil(1.0 / float(dz_m))))
    peaks, _ = signal.find_peaks(autocorrelation[peak_start:], prominence=0.02)
    if peaks.size:
        first_peak_index = int(peaks[0] + peak_start)
        first_peak_lag_m = float(lag_m[first_peak_index])
        first_peak_amplitude = float(autocorrelation[first_peak_index])
    else:
        first_peak_lag_m = np.nan
        first_peak_amplitude = np.nan

    return {
        "frequency_cycles_per_m": frequency,
        "power_probability": power_probability,
        "lag_m": lag_m[keep],
        "autocorrelation": autocorrelation[keep],
        "crossing_interval_m": crossing_interval_m,
        "spectral_entropy": spectral_entropy,
        "strongest_peak_fraction": peak_fraction,
        "first_secondary_peak_lag_m": first_peak_lag_m,
        "first_secondary_peak_amplitude": first_peak_amplitude,
    }


def rms(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    return float(np.sqrt(np.mean(values**2))) if values.size else np.nan


def result_metrics(well, setting, result, synthetic_full, synthetic_recoverable):
    support = result["support"] & np.isfinite(synthetic_full) & np.isfinite(synthetic_recoverable)
    residual = result["high_frequency_residual"]
    periodicity = periodicity_profile(
        residual,
        result["support"],
        well["dz_m"],
        max_lag_m=4.0 * well["tuning_scale_m"],
    )
    synthetic_residual = synthetic_full - synthetic_recoverable
    forward_supported_samples = int(np.count_nonzero(support))
    if forward_supported_samples >= 2:
        corr = float(np.corrcoef(synthetic_full[support], synthetic_recoverable[support])[0, 1])
        residual_synthetic_rms_ratio = rms(synthetic_residual[support]) / max(rms(synthetic_full[support]), 1e-12)
        forward_status = "ok"
    else:
        corr = np.nan
        residual_synthetic_rms_ratio = np.nan
        forward_status = "unavailable_no_common_support"

    edge_values = []
    interior_values = []
    edge_samples = int(np.ceil(result["fwhm_m"] / well["dz_m"]))
    for run in finite_runs(result["support"]):
        length = run.stop - run.start
        width = min(edge_samples, length // 3)
        if width <= 0:
            continue
        edge_values.extend(residual[run.start : run.start + width])
        edge_values.extend(residual[run.stop - width : run.stop])
        if length > 2 * width:
            interior_values.extend(residual[run.start + width : run.stop - width])
    edge_ratio = rms(edge_values) / max(rms(interior_values), 1e-12)

    valid_residual = residual[result["support"]]
    crossing_intervals = periodicity["crossing_interval_m"] if periodicity is not None else np.asarray([])
    return {
        "well_name": well["well_name"],
        "method": "gaussian_scale_space",
        "setting": setting,
        "tie_corr": well["tie_corr"],
        "reference_frequency_hz": reference_frequency_hz,
        "median_vp_mps": well["median_vp_mps"],
        "tuning_scale_m": well["tuning_scale_m"],
        "fwhm_m": result["fwhm_m"],
        "longest_valid_run_fwhm_ratio": result["longest_valid_run_fwhm_ratio"],
        "edge_dominated": result["edge_dominated"],
        "supported_samples": int(np.count_nonzero(result["support"])),
        "forward_status": forward_status,
        "forward_supported_samples": forward_supported_samples,
        "reconstruction_max_abs": float(
            np.nanmax(np.abs(result["reconstructed_log_ai"][result["support"]] - well["log_ai"][result["support"]]))
        ),
        "residual_rms": rms(valid_residual),
        "residual_mad": float(np.median(np.abs(valid_residual - np.median(valid_residual)))),
        "residual_abs_p95": float(np.percentile(np.abs(valid_residual), 95.0)),
        "residual_synthetic_rms_ratio": residual_synthetic_rms_ratio,
        "recoverable_synthetic_corr": corr,
        "edge_interior_energy_ratio": float(edge_ratio),
        "spectral_entropy": periodicity["spectral_entropy"] if periodicity is not None else np.nan,
        "strongest_spectral_peak_fraction": (
            periodicity["strongest_peak_fraction"] if periodicity is not None else np.nan
        ),
        "autocorr_secondary_peak_lag_m": (
            periodicity["first_secondary_peak_lag_m"] if periodicity is not None else np.nan
        ),
        "autocorr_secondary_peak_amplitude": (
            periodicity["first_secondary_peak_amplitude"] if periodicity is not None else np.nan
        ),
        "zero_crossing_interval_cv": (
            float(np.std(crossing_intervals) / np.mean(crossing_intervals))
            if crossing_intervals.size >= 2 and np.mean(crossing_intervals) > 0.0
            else np.nan
        ),
    }

## 3. 固定图件

每口井输出：

- `overview.png`：全曲线与三个 recoverable/residual 轨道；
- `zoom.png`：原井梯度能量较高的固定长度窗口；
- `periodicity.png`：功率谱、自相关和过零间距；
- `<well>.npz`：曲线、支持区、层位及三档分解结果。

图件中的三个 residual 使用同一个横轴范围，避免缩放制造“更平滑”的错觉。


In [ ]:
SETTING_COLORS = {
    "G075": "#1f77b4",
    "G100": "#ff7f0e",
    "G150": "#2ca02c",
}


def plot_limits(well):
    horizons = np.asarray(list(well["horizons"].values()), dtype=np.float64)
    finite_depth = well["tvdss_m"][well["valid"]]
    lower = max(float(np.min(finite_depth)), float(np.min(horizons) - PLOT_CONTEXT_M))
    upper = min(float(np.max(finite_depth)), float(np.max(horizons) + PLOT_CONTEXT_M))
    return lower, upper


def draw_horizons(axis, well):
    for name, depth in well["horizons"].items():
        axis.axhline(depth, color="0.45", linestyle="--", linewidth=0.8)
        axis.text(
            0.02,
            depth,
            name,
            transform=axis.get_yaxis_transform(),
            va="bottom",
            fontsize=7,
            color="0.35",
        )


def choose_zoom_window(well):
    depth = well["tvdss_m"]
    values = well["log_ai"]
    lower, upper = plot_limits(well)
    in_target = well["valid"] & (depth >= lower) & (depth <= upper)
    window_samples = max(5, int(round(ZOOM_WINDOW_M / well["dz_m"])))
    gradient = np.zeros(values.shape, dtype=np.float64)
    for run in finite_runs(well["valid"]):
        if run.stop - run.start >= 3:
            gradient[run] = np.gradient(values[run], depth[run])
    energy = signal.convolve(
        np.where(in_target, gradient**2, 0.0),
        np.ones(window_samples),
        mode="same",
    )
    support_count = signal.convolve(
        in_target.astype(np.float64),
        np.ones(window_samples),
        mode="same",
    )
    energy[support_count < 0.8 * window_samples] = -np.inf
    if not np.any(np.isfinite(energy)):
        return lower, min(upper, lower + ZOOM_WINDOW_M)
    center = float(depth[int(np.nanargmax(energy))])
    half = 0.5 * ZOOM_WINDOW_M
    return max(lower, center - half), min(upper, center + half)


def plot_well_figures(well, results):
    well_dir = OUTPUT_DIR / "figures" / well["well_name"]
    well_dir.mkdir(parents=True, exist_ok=True)
    depth = well["tvdss_m"]
    lower, upper = plot_limits(well)
    view = (depth >= lower) & (depth <= upper)

    residual_values = np.concatenate(
        [item["high_frequency_residual"][view & item["support"]] for item in results.values()]
    )
    residual_limit = float(np.percentile(np.abs(residual_values), 99.0))
    full_values = well["log_ai"][view & well["valid"]]
    full_limits = np.percentile(full_values, [1.0, 99.0])

    fig, axes = plt.subplots(1, 4, figsize=(13.5, 8.5), sharey=True, constrained_layout=True)
    axes[0].plot(well["log_ai"], depth, color="black", linewidth=0.8, label="full log-AI")
    for setting, item in results.items():
        axes[0].plot(
            item["recoverable_log_ai"],
            depth,
            color=SETTING_COLORS[setting],
            linewidth=1.0,
            label=f"{setting} ({item['fwhm_m']:.1f} m)",
        )
    axes[0].set_xlim(*full_limits)
    axes[0].set_xlabel("log-AI")
    axes[0].set_ylabel("TVDSS (m)")
    axes[0].legend(fontsize=7, loc="best")

    for axis, (setting, item) in zip(axes[1:], results.items()):
        axis.plot(
            item["high_frequency_residual"],
            depth,
            color=SETTING_COLORS[setting],
            linewidth=0.75,
        )
        axis.axvline(0.0, color="0.4", linewidth=0.7)
        axis.set_xlim(-residual_limit, residual_limit)
        axis.set_xlabel(f"{setting} residual")
        axis.set_title(f"FWHM={item['fwhm_m']:.1f} m")

    for axis in axes:
        axis.set_ylim(upper, lower)
        draw_horizons(axis, well)
    fig.suptitle(
        f"{well['well_name']} | tie corr={well['tie_corr']:.3f} | T={well['tuning_scale_m']:.1f} m",
        fontsize=12,
    )
    overview_path = well_dir / "overview.png"
    fig.savefig(overview_path, bbox_inches="tight")
    plt.close(fig)

    zoom_lower, zoom_upper = choose_zoom_window(well)
    fig, axes = plt.subplots(3, 2, figsize=(10.5, 10.5), sharey=True, constrained_layout=True)
    for row, (setting, item) in enumerate(results.items()):
        left, right = axes[row]
        left.plot(well["log_ai"], depth, color="black", linewidth=0.9, label="full")
        left.plot(
            item["recoverable_log_ai"],
            depth,
            color=SETTING_COLORS[setting],
            linewidth=1.1,
            label="recoverable",
        )
        left.set_xlabel("log-AI")
        left.set_ylabel("TVDSS (m)")
        left.legend(fontsize=7)
        right.plot(
            item["high_frequency_residual"],
            depth,
            color=SETTING_COLORS[setting],
            linewidth=0.9,
        )
        right.axvline(0.0, color="0.4", linewidth=0.7)
        right.set_xlim(-residual_limit, residual_limit)
        right.set_xlabel(f"{setting} residual")
        for axis in (left, right):
            axis.set_ylim(zoom_upper, zoom_lower)
            draw_horizons(axis, well)
    fig.suptitle(f"{well['well_name']} | fixed {zoom_upper - zoom_lower:.1f} m zoom")
    zoom_path = well_dir / "zoom.png"
    fig.savefig(zoom_path, bbox_inches="tight")
    plt.close(fig)

    fig, axes = plt.subplots(3, 3, figsize=(13.0, 9.5), constrained_layout=True)
    for row, (setting, item) in enumerate(results.items()):
        profile = item["periodicity"]
        if profile is None:
            continue
        frequency = profile["frequency_cycles_per_m"]
        wavelength = 1.0 / frequency
        order = np.argsort(wavelength)
        axes[row, 0].plot(
            wavelength[order],
            profile["power_probability"][order],
            color=SETTING_COLORS[setting],
        )
        axes[row, 0].set_xscale("log")
        axes[row, 0].set_xlim(0.2, 300.0)
        axes[row, 0].set_xlabel("vertical wavelength (m)")
        axes[row, 0].set_ylabel("normalized power")
        axes[row, 0].set_title(f"{setting} spectrum")

        axes[row, 1].plot(
            profile["lag_m"],
            profile["autocorrelation"],
            color=SETTING_COLORS[setting],
        )
        axes[row, 1].axhline(0.0, color="0.4", linewidth=0.7)
        axes[row, 1].set_xlabel("lag (m)")
        axes[row, 1].set_ylabel("autocorrelation")

        intervals = profile["crossing_interval_m"]
        if intervals.size:
            axes[row, 2].hist(
                intervals,
                bins=min(30, max(5, int(np.sqrt(intervals.size)))),
                color=SETTING_COLORS[setting],
                alpha=0.8,
            )
        axes[row, 2].set_xlabel("zero-crossing interval (m)")
        axes[row, 2].set_ylabel("count")
    fig.suptitle(f"{well['well_name']} | periodicity diagnostics")
    periodicity_path = well_dir / "periodicity.png"
    fig.savefig(periodicity_path, bbox_inches="tight")
    plt.close(fig)

    return {
        "overview": str(overview_path),
        "zoom": str(zoom_path),
        "periodicity": str(periodicity_path),
    }

In [ ]:
all_results = {}
metric_rows = []
figure_paths = {}
well_artifacts = {}

for well_name, well in wells.items():
    synthetic_full = projected_forward_log_ai(well["tvdss_m"], well["log_ai"], well["valid"], well["dz_m"])
    settings = {}
    for setting, multiplier in FWHM_MULTIPLIERS.items():
        result = gaussian_decomposition(
            well,
            fwhm_m=multiplier * well["tuning_scale_m"],
        )
        synthetic_recoverable = projected_forward_log_ai(
            well["tvdss_m"],
            result["recoverable_log_ai"],
            result["support"],
            well["dz_m"],
        )
        result["synthetic_full"] = synthetic_full
        result["synthetic_recoverable"] = synthetic_recoverable
        result["synthetic_residual"] = synthetic_full - synthetic_recoverable
        result["periodicity"] = periodicity_profile(
            result["high_frequency_residual"],
            result["support"],
            well["dz_m"],
            max_lag_m=4.0 * well["tuning_scale_m"],
        )
        metric_row = result_metrics(
            well,
            setting,
            result,
            synthetic_full,
            synthetic_recoverable,
        )
        metric_rows.append(metric_row)
        if metric_row["forward_status"] != "ok":
            print(f"warning: {well_name} {setting} forward diagnostic unavailable ({metric_row['forward_status']})")
        settings[setting] = result

    all_results[well_name] = settings
    figure_paths[well_name] = plot_well_figures(well, settings)

    artifact_path = OUTPUT_DIR / "wells" / f"{well_name}.npz"
    payload = {
        "tvdss_m": well["tvdss_m"],
        "well_log_ai": well["log_ai"],
        "well_valid": well["valid"],
        "horizon_names": np.asarray(list(well["horizons"].keys()), dtype="U64"),
        "horizon_tvdss_m": np.asarray(list(well["horizons"].values()), dtype=np.float64),
    }
    for setting, result in settings.items():
        prefix = setting.lower()
        payload[f"{prefix}_fwhm_m"] = np.asarray(result["fwhm_m"])
        payload[f"{prefix}_support"] = result["support"]
        payload[f"{prefix}_recoverable_log_ai"] = result["recoverable_log_ai"]
        payload[f"{prefix}_high_frequency_residual"] = result["high_frequency_residual"]
        payload[f"{prefix}_synthetic_full"] = result["synthetic_full"]
        payload[f"{prefix}_synthetic_recoverable"] = result["synthetic_recoverable"]
        payload[f"{prefix}_synthetic_residual"] = result["synthetic_residual"]
    np.savez_compressed(artifact_path, **payload)
    well_artifacts[well_name] = str(artifact_path)
    print(f"completed {well_name}")

metrics = pd.DataFrame(metric_rows).sort_values(["well_name", "setting"])
metrics_path = OUTPUT_DIR / "metrics.csv"
metrics.to_csv(metrics_path, index=False)

review_columns = [
    "well_name",
    "method",
    "setting",
    "geological_texture",
    "fixed_wavelength_striping",
    "thickness_amplitude_variation",
    "horizon_edge_artifact",
    "long_trend_leakage",
    "residual_forward_too_strong",
    "usable_as_prior",
    "notes",
]
human_review = metrics[["well_name", "method", "setting"]].copy()
for column in review_columns[3:]:
    human_review[column] = ""
review_path = OUTPUT_DIR / "human_review.csv"
human_review.to_csv(review_path, index=False)

manifest = {
    "schema": "well_log_gaussian_residual_experiment_v1",
    "run_id": RUN_ID,
    "status": ("completed_with_warnings" if metrics["forward_status"].ne("ok").any() else "completed"),
    "sample_domain": "depth",
    "sample_unit": "m",
    "depth_basis": "tvdss",
    "trusted_wells": list(TRUSTED_WELLS),
    "diagnostic_wells_included": list(DIAGNOSTIC_WELLS) if INCLUDE_DIAGNOSTIC_WELLS else [],
    "inputs": {
        "wavelet_batch_metrics": str(METRICS_FILE),
        "well_tops": str(WELL_TOPS_FILE),
        "forward_inputs": str(FORWARD_INPUTS_FILE),
        "wavelet": str(wavelet_path),
    },
    "wavelet_frequency_summary": wavelet_summary,
    "reference_frequency_statistic": REFERENCE_FREQUENCY_STATISTIC,
    "reference_frequency_hz": reference_frequency_hz,
    "fwhm_multipliers": FWHM_MULTIPLIERS,
    "well_artifacts": well_artifacts,
    "figures": figure_paths,
    "metrics": str(metrics_path),
    "human_review": str(review_path),
}
manifest_path = OUTPUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

display(
    metrics[
        [
            "well_name",
            "setting",
            "tuning_scale_m",
            "fwhm_m",
            "edge_dominated",
            "forward_status",
            "residual_rms",
            "residual_synthetic_rms_ratio",
            "recoverable_synthetic_corr",
            "spectral_entropy",
            "strongest_spectral_peak_fraction",
            "zero_crossing_interval_cv",
        ]
    ]
)
print(f"Manifest: {manifest_path}")
print(f"Human review template: {review_path}")

In [ ]:
# 汇总图：每行一口井，第一列为 full/recoverable，后三列为同尺度 residual。
fig, axes = plt.subplots(
    len(TRUSTED_WELLS),
    4,
    figsize=(14.0, 3.2 * len(TRUSTED_WELLS)),
    constrained_layout=True,
    squeeze=False,
)
for row, well_name in enumerate(TRUSTED_WELLS):
    well = wells[well_name]
    results = all_results[well_name]
    depth = well["tvdss_m"]
    lower, upper = plot_limits(well)
    view = (depth >= lower) & (depth <= upper)
    residual_values = np.concatenate(
        [item["high_frequency_residual"][view & item["support"]] for item in results.values()]
    )
    residual_limit = float(np.percentile(np.abs(residual_values), 99.0))

    axes[row, 0].plot(well["log_ai"], depth, color="black", linewidth=0.65)
    for setting, item in results.items():
        axes[row, 0].plot(
            item["recoverable_log_ai"],
            depth,
            color=SETTING_COLORS[setting],
            linewidth=0.8,
        )
    axes[row, 0].set_ylabel(f"{well_name}\nTVDSS (m)")
    if row == 0:
        axes[row, 0].set_title("full + recoverable")

    for column, (setting, item) in enumerate(results.items(), start=1):
        axes[row, column].plot(
            item["high_frequency_residual"],
            depth,
            color=SETTING_COLORS[setting],
            linewidth=0.65,
        )
        axes[row, column].axvline(0.0, color="0.5", linewidth=0.5)
        axes[row, column].set_xlim(-residual_limit, residual_limit)
        if row == 0:
            axes[row, column].set_title(setting)
    for axis in axes[row]:
        axis.set_ylim(upper, lower)
        draw_horizons(axis, well)
        axis.set_xlabel("log-AI")
atlas_path = OUTPUT_DIR / "figures" / "trusted_wells_atlas.png"
fig.savefig(atlas_path, bbox_inches="tight")
plt.close(fig)

display(Image(filename=str(atlas_path)))
print(atlas_path)

## 4. 人工检查顺序

1. 先看 `figures/trusted_wells_atlas.png`，判断某一档是否在所有井上都出现相同固定波长。
2. 再逐井看 `overview.png` 和 `zoom.png`，判断残差是否跟随真实井曲线的局部薄互层和幅度变化。
3. 最后看 `periodicity.png`，确认肉眼观察到的条纹是否对应窄谱峰、强自相关峰或过于集中的过零间距。
4. 在 `human_review.csv` 中逐井、逐 setting 记录评价。

第一轮只选择“值得进入第二轮比较的形态范围”，不选择最终高频先验。
